<a href="https://colab.research.google.com/github/e23323-dot/Statistical-Learning-e23323/blob/main/Assignment_7C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q1. Bayesian Estimation of a User Ability Parameter from Item Responses")
print("="*70)

print("""
1. Visualizing the Mechanics:

P(Y_i = 1 | Θ = θ) = p_i(θ) = 1 / (1 + e^{-a_i(θ - b_i)})

where:
- a_i > 0 is the discrimination parameter
- b_i is the difficulty parameter

Interpretation:
- Increasing b_i shifts the curve right (harder item, requires higher ability)
- Increasing a_i makes the curve steeper (more discriminating between abilities)
""")

theta_range = np.linspace(-4, 4, 500)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Different b values (a=1)", "Different a values (b=0)"))

for b in [-1, 0, 1]:
    p = 1/(1 + np.exp(-1*(theta_range - b)))
    fig.add_trace(go.Scatter(x=theta_range, y=p, mode='lines', name=f'b = {b}'), row=1, col=1)

for a in [0.5, 1, 2]:
    p = 1/(1 + np.exp(-a*(theta_range - 0)))
    fig.add_trace(go.Scatter(x=theta_range, y=p, mode='lines', name=f'a = {a}'), row=1, col=2)

fig.update_layout(height=400, width=800, title_text="Item Response Functions")
fig.show()

print("""
2. Sequential Likelihood Contribution:

For a single response y_k:
L(y_k | θ) = [p_k(θ)]^{y_k} [1 - p_k(θ)]^{1 - y_k}

where y_k ∈ {0, 1} and p_k(θ) = 1 / (1 + e^{-a_k(θ - b_k)})

For the running history vector y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = ∏_{i=1}^k [p_i(θ)]^{y_i} [1 - p_i(θ)]^{1 - y_i}

3. Mathematical Formulation of the Running Update:

f_{Θ | Y^(k)}(θ | y^(k)) ∝ L(y_k | θ) · f_{Θ | Y^(k-1)}(θ | y^(k-1))

This recursive relationship uses the posterior from step (k-1) as the prior for step k.

4. Dynamic Shifting:

When y_k = 1 (correct answer) to a highly difficult item (large b_k):
- p_k(θ) is small for low θ and large for high θ
- The likelihood L(1|θ) = p_k(θ) acts as an increasing function of θ
- Multiplying the prior by this increasing function shifts probability mass rightward
- The posterior peak moves toward higher θ values

When y_k = 0 (incorrect answer):
- L(0|θ) = 1 - p_k(θ) acts as a decreasing function of θ
- The posterior peak shifts leftward toward lower θ values

5. Tracking Certainty and Sharpness:

The discrimination parameter a_k controls the likelihood function shape:

When a_k is very large (a_k → ∞):
- p_k(θ) approaches a step function at θ = b_k
- The likelihood becomes highly concentrated around θ = b_k
- Posterior variance decreases significantly (high certainty)
- The update provides strong information about the user's ability

When a_k is very small (a_k → 0):
- p_k(θ) approaches 0.5 for all θ
- The likelihood provides almost no information
- Posterior variance remains large (low certainty)
- The update provides weak information about the user's ability

6. Numerical Implementation of a Running Grid:

Algorithm for maintaining posterior on a fixed grid:

Step 1: Define grid
- Choose M grid points: θ_1, θ_2, ..., θ_M over a reasonable range (e.g., [-4, 4])
- Compute grid spacing: Δθ = θ_2 - θ_1

Step 2: Initialize prior
- For each grid point j, compute prior density:
  f_0(θ_j) = (1/√(2π)) · exp(-θ_j²/2)
- Normalize using trapezoidal rule:
  f_0(θ_j) = f_0(θ_j) / [Σ_{m=1}^M f_0(θ_m) · Δθ]

Step 3: Sequential update for each k = 1, 2, ..., n:
   a. Compute likelihood at each grid point:
      L(y_k | θ_j) = [p_k(θ_j)]^{y_k} [1 - p_k(θ_j)]^{1 - y_k}
      where p_k(θ_j) = 1 / (1 + e^{-a_k(θ_j - b_k)})

   b. Compute unnormalized posterior:
      g(θ_j) = L(y_k | θ_j) · f_{k-1}(θ_j)

   c. Normalize using trapezoidal rule:
      Z_k = Σ_{m=1}^M g(θ_m) · Δθ
      f_k(θ_j) = g(θ_j) / Z_k

   d. Store point estimates:
      Posterior Mean: θ̂_Bayes^(k) = Σ_{j=1}^M θ_j · f_k(θ_j) · Δθ
      MAP: θ̂_MAP^(k) = argmax_{θ_j} f_k(θ_j)
""")

def simulate_irt(n=20, theta_true=0.75):
    np.random.seed(42)
    a_vals = np.random.uniform(0.5, 2.0, n)
    b_vals = np.random.normal(0, 1, n)

    theta_grid = np.linspace(-4, 4, 200)
    delta_theta = theta_grid[1] - theta_grid[0]
    prior = np.exp(-0.5*theta_grid**2)/np.sqrt(2*np.pi)
    prior /= np.trapezoid(prior, theta_grid)

    posterior_mean_est = []
    map_est = []

    for k in range(n):
        p_true = 1/(1 + np.exp(-a_vals[k]*(theta_true - b_vals[k])))
        y_k = 1 if np.random.random() < p_true else 0

        p_theta = 1/(1 + np.exp(-a_vals[k]*(theta_grid - b_vals[k])))
        likelihood = (p_theta**y_k) * ((1 - p_theta)**(1 - y_k))

        posterior = prior * likelihood
        posterior /= np.trapezoid(posterior, theta_grid)

        mean_est = np.trapezoid(theta_grid * posterior, theta_grid)
        map_est_val = theta_grid[np.argmax(posterior)]

        posterior_mean_est.append(mean_est)
        map_est.append(map_est_val)

        prior = posterior

    return posterior_mean_est, map_est

post_mean, map_est = simulate_irt()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(21)), y=[0.75]*21, mode='lines', name='θ_true = 0.75', line=dict(dash='dash')))
fig2.add_trace(go.Scatter(x=list(range(1,21)), y=post_mean, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=list(range(1,21)), y=map_est, mode='lines+markers', name='MAP'))
fig2.update_layout(title='IRT: Posterior Mean and MAP vs True Ability', xaxis_title='Step k', yaxis_title='θ Estimate', height=500, width=800)
fig2.show()

print("""
7. Evaluating Convergence over the Timeline:

Analysis:
As k increases, both estimators converge toward θ_true = 0.75.
The initial prior N(0, 1) is quickly overwhelmed by observed responses.
After approximately 10 items, the platform has high confidence in the user's ability.
The narrowing gap between estimates and θ_true indicates increasing measurement precision.
""")

Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

1. Visualizing the Mechanics:

P(Y_i = 1 | Θ = θ) = p_i(θ) = 1 / (1 + e^{-a_i(θ - b_i)})

where:
- a_i > 0 is the discrimination parameter
- b_i is the difficulty parameter

Interpretation:
- Increasing b_i shifts the curve right (harder item, requires higher ability)
- Increasing a_i makes the curve steeper (more discriminating between abilities)




2. Sequential Likelihood Contribution:

For a single response y_k:
L(y_k | θ) = [p_k(θ)]^{y_k} [1 - p_k(θ)]^{1 - y_k}

where y_k ∈ {0, 1} and p_k(θ) = 1 / (1 + e^{-a_k(θ - b_k)})

For the running history vector y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = ∏_{i=1}^k [p_i(θ)]^{y_i} [1 - p_i(θ)]^{1 - y_i}

3. Mathematical Formulation of the Running Update:

f_{Θ | Y^(k)}(θ | y^(k)) ∝ L(y_k | θ) · f_{Θ | Y^(k-1)}(θ | y^(k-1))

This recursive relationship uses the posterior from step (k-1) as the prior for step k.

4. Dynamic Shifting:

When y_k = 1 (correct answer) to a highly difficult item (large b_k):
- p_k(θ) is small for low θ and large for high θ
- The likelihood L(1|θ) = p_k(θ) acts as an increasing function of θ
- Multiplying the prior by this increasing function shifts probability mass rightward
- The posterior peak moves toward higher θ values

When y_k = 0 (incorrect answer):
- L(0|θ) = 1 - p_k(θ) acts as a decreasing function of θ
- The posterior peak shifts leftward toward lo


7. Evaluating Convergence over the Timeline:

Analysis:
As k increases, both estimators converge toward θ_true = 0.75.
The initial prior N(0, 1) is quickly overwhelmed by observed responses.
After approximately 10 items, the platform has high confidence in the user's ability.
The narrowing gap between estimates and θ_true indicates increasing measurement precision.



In [4]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q2. Bayesian Tracking of Click-Through Rates (CTR)")
print("="*70)

print("""
1. Structural Probability and Properties:

Beta Distribution PDF:
f_Θ(θ) = [θ^(α-1) (1-θ)^(β-1)] / B(α, β)

where B(α, β) = Γ(α)Γ(β) / Γ(α+β) is the Beta function.

Properties:
- Mean: E[Θ] = α / (α + β)
- Mode: (α - 1) / (α + β - 2) for α, β > 1
- Variance: αβ / [(α + β)²(α + β + 1)]

Interpretation of parameter pairs:
- (α=1, β=1): Uniform (uninformative) - all θ equally likely
- (α=2, β=8): Right-skewed - higher probability of smaller θ (low CTR)
- (α=8, β=2): Left-skewed - higher probability of larger θ (high CTR)
""")

theta_range = np.linspace(0.001, 0.999, 500)

fig = go.Figure()
params = [(1, 1, 'Uninformative'), (2, 8, 'Right-skewed'), (8, 2, 'Left-skewed')]
for alpha, beta_val, label in params:
    pdf = beta.pdf(theta_range, alpha, beta_val)
    fig.add_trace(go.Scatter(x=theta_range, y=pdf, mode='lines', name=f'Beta({alpha},{beta_val})'))

fig.update_layout(title='Beta Distributions', xaxis_title='θ', yaxis_title='Density', height=400, width=800)
fig.show()

print("""
2. Sequential Likelihood and Joint History:

For a single response y_k:
L(y_k | θ) = θ^{y_k} (1-θ)^{1-y_k}

For the running history y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = θ^{Σ_{i=1}^k y_i} (1-θ)^{k - Σ_{i=1}^k y_i}

3. Closed-Form Analytical Updates (Conjugacy):

Using Bayes' Theorem:
f(θ | y^(k)) ∝ L(y^(k) | θ) · f(θ)

= θ^{Σ y_i} (1-θ)^{k-Σ y_i} · θ^{α₀-1} (1-θ)^{β₀-1}

= θ^{α₀ + Σ y_i - 1} (1-θ)^{β₀ + k - Σ y_i - 1}

Therefore, the posterior remains in the Beta family:

Θ | Y^(k) ~ Beta(α_k, β_k)

where the update rules are:
α_k = α_{k-1} + y_k
β_k = β_{k-1} + (1 - y_k)

Posterior Mean:
E[Θ | Y^(k) = y^(k)] = α_k / (α_k + β_k)

4. Dynamic Shifting Mechanics:

When y_k = 1 (click observed):
- α_k = α_{k-1} + 1
- The mean increases: E[θ] = α_k/(α_k+β_k) > α_{k-1}/(α_{k-1}+β_{k-1})
- The distribution shifts right (toward higher θ)

When y_k = 0 (no click observed):
- β_k = β_{k-1} + 1
- The mean decreases
- The distribution shifts left (toward lower θ)

Contrast with non-conjugate setups:
- In conjugate models (Beta-Binomial), updates are exact and closed-form
- No numerical integration required; O(1) computational cost per step
- In non-conjugate models (like 2PL IRT), numerical grid integration is required
- Non-conjugate requires O(M) operations per step where M is grid size

5. Running Point Estimators:

Posterior Mean (Bayes estimate under squared error loss):
θ̂_Bayes^(k) = E[Θ | Y^(k)] = α_k / (α_k + β_k)

Maximum A Posteriori (MAP) estimate:
θ̂_MAP^(k) = argmax_θ f(θ | y^(k)) = (α_k - 1) / (α_k + β_k - 2)
provided α_k > 1 and β_k > 1

6. Performance Tracking and Convergence Analysis:
""")

def simulate_ctr(n=100, theta_true=0.35, alpha0=1, beta0=1):
    np.random.seed(42)
    alpha_k, beta_k = alpha0, beta0
    mean_est = []
    map_est = []

    for k in range(1, n+1):
        y_k = 1 if np.random.random() < theta_true else 0
        alpha_k += y_k
        beta_k += (1 - y_k)

        mean_est.append(alpha_k/(alpha_k+beta_k))
        if alpha_k > 1 and beta_k > 1:
            map_est.append((alpha_k-1)/(alpha_k+beta_k-2))
        else:
            map_est.append(alpha_k/(alpha_k+beta_k))

    return mean_est, map_est

mean_est, map_est = simulate_ctr()

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(101)), y=[0.35]*101, mode='lines', name='θ_true = 0.35', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=list(range(1,101)), y=mean_est, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=list(range(1,101)), y=map_est, mode='lines', name='MAP'))
fig.update_layout(title='CTR: Posterior Mean and MAP vs True CTR', xaxis_title='Impression k', yaxis_title='θ Estimate', height=500, width=800)
fig.show()

print("""
Analysis:
Both estimators converge to θ_true = 0.35 as k increases.

Behavior over time:
- Initial uniform prior (α₀ = β₀ = 1) provides no information
- Early steps: estimates fluctuate significantly due to limited data
- After approximately 50 impressions: estimates stabilize near true value
- As k approaches 100: estimates are tightly concentrated around 0.35

Implications for accumulation of evidence:
- The prior is quickly overwhelmed by observed data
- With more impressions, posterior variance decreases
- The platform gains confidence in its CTR estimate
- The choice of uniform prior ensures unbiased learning from data
""")

Q2. Bayesian Tracking of Click-Through Rates (CTR)

1. Structural Probability and Properties:

Beta Distribution PDF:
f_Θ(θ) = [θ^(α-1) (1-θ)^(β-1)] / B(α, β)

where B(α, β) = Γ(α)Γ(β) / Γ(α+β) is the Beta function.

Properties:
- Mean: E[Θ] = α / (α + β)
- Mode: (α - 1) / (α + β - 2) for α, β > 1
- Variance: αβ / [(α + β)²(α + β + 1)]

Interpretation of parameter pairs:
- (α=1, β=1): Uniform (uninformative) - all θ equally likely
- (α=2, β=8): Right-skewed - higher probability of smaller θ (low CTR)
- (α=8, β=2): Left-skewed - higher probability of larger θ (high CTR)




2. Sequential Likelihood and Joint History:

For a single response y_k:
L(y_k | θ) = θ^{y_k} (1-θ)^{1-y_k}

For the running history y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = θ^{Σ_{i=1}^k y_i} (1-θ)^{k - Σ_{i=1}^k y_i}

3. Closed-Form Analytical Updates (Conjugacy):

Using Bayes' Theorem:
f(θ | y^(k)) ∝ L(y^(k) | θ) · f(θ)

= θ^{Σ y_i} (1-θ)^{k-Σ y_i} · θ^{α₀-1} (1-θ)^{β₀-1}

= θ^{α₀ + Σ y_i - 1} (1-θ)^{β₀ + k - Σ y_i - 1}

Therefore, the posterior remains in the Beta family:

Θ | Y^(k) ~ Beta(α_k, β_k)

where the update rules are:
α_k = α_{k-1} + y_k
β_k = β_{k-1} + (1 - y_k)

Posterior Mean:
E[Θ | Y^(k) = y^(k)] = α_k / (α_k + β_k)

4. Dynamic Shifting Mechanics:

When y_k = 1 (click observed):
- α_k = α_{k-1} + 1
- The mean increases: E[θ] = α_k/(α_k+β_k) > α_{k-1}/(α_{k-1}+β_{k-1})
- The distribution shifts right (toward higher θ)

When y_k = 0 (no click observed):
- β_k = β_{k-1} + 1
- The mean decreases
- The distribution shifts left (toward lower θ)

Contrast with non-conjuga


Analysis:
Both estimators converge to θ_true = 0.35 as k increases.

Behavior over time:
- Initial uniform prior (α₀ = β₀ = 1) provides no information
- Early steps: estimates fluctuate significantly due to limited data
- After approximately 50 impressions: estimates stabilize near true value
- As k approaches 100: estimates are tightly concentrated around 0.35

Implications for accumulation of evidence:
- The prior is quickly overwhelmed by observed data
- With more impressions, posterior variance decreases
- The platform gains confidence in its CTR estimate
- The choice of uniform prior ensures unbiased learning from data

